In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/imuhammad/course-reviews-on-coursera/Coursera_reviews.csv
/kaggle/input/datasets/imuhammad/course-reviews-on-coursera/Coursera_courses.csv
/kaggle/input/datasets/ayaanfrngl/glove-100d/glove.6B.100d.txt


In [2]:
import pandas as pd
import numpy as np
import re

In [3]:
df = pd.read_csv("/kaggle/input/datasets/imuhammad/course-reviews-on-coursera/Coursera_reviews.csv")
df.head()

,reviews,reviewers,date_reviews,rating,course_id
0,"Pretty dry, but I was able to pass with just t...",By Robert S,"Feb 12, 2020",4,google-cbrs-cpi-training
1,would be a better experience if the video and ...,By Gabriel E R,"Sep 28, 2020",4,google-cbrs-cpi-training
2,Information was perfect! The program itself wa...,By Jacob D,"Apr 08, 2020",4,google-cbrs-cpi-training
3,A few grammatical mistakes on test made me do ...,By Dale B,"Feb 24, 2020",4,google-cbrs-cpi-training
4,Excellent course and the training provided was...,By Sean G,"Jun 18, 2020",4,google-cbrs-cpi-training


In [4]:
def convert_rating(r):
    if r <= 2:
        return "negative"
    elif r == 3:
        return "neutral"
    else:
        return "positive"

df["sentiment"] = df["rating"].apply(convert_rating)
df["sentiment"].value_counts()

sentiment
positive    1372866
neutral       48303
negative      33542
Name: count, dtype: int64

In [5]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df["cleaned_text"] = df["reviews"].apply(clean_text)

In [6]:
from sklearn.model_selection import train_test_split

X = df["cleaned_text"]
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [7]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Hyperparameters
max_words = 20000
max_len = 100

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')

print("Train shape:", X_train_pad.shape)
print("Test shape:", X_test_pad.shape)

2026-02-25 07:46:30.803211: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772005590.969884      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772005591.015019      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772005591.407190      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772005591.407217      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772005591.407219      24 computation_placer.cc:177] computation placer alr

Train shape: (1163768, 100)
Test shape: (290943, 100)


In [8]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

print("Classes:", le.classes_)

Classes: ['negative' 'neutral' 'positive']


In [9]:
import numpy as np

embedding_dim = 100
embeddings_index = {}

with open("/kaggle/input/datasets/ayaanfrngl/glove-100d/glove.6B.100d.txt", encoding="utf8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype="float32")
        embeddings_index[word] = vector

print("Loaded %s word vectors." % len(embeddings_index))

Loaded 400000 word vectors.


In [10]:
word_index = tokenizer.word_index

embedding_matrix = np.zeros((max_words, embedding_dim))

for word, i in word_index.items():
    if i < max_words:
        embedding_vector = embeddings_index.get(word)
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector

print("Embedding matrix ready")

Embedding matrix ready


In [11]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential()

# Embedding Layer (with GloVe weights)
model.add(
    Embedding(
        input_dim=max_words,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        input_length=max_len,
        trainable=False  # keep GloVe fixed
    )
)

# LSTM Layer
model.add(LSTM(128))

# Dropout (reduce overfitting)
model.add(Dropout(0.5))

# Dense layers
model.add(Dense(64, activation='relu'))
model.add(Dense(3, activation='softmax'))  # 3 classes

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
I0000 00:00:1772005663.882407      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1772005663.888330      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │     2,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,000,000 (7.63 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 2,000,000 (7.63 MB)

In [12]:
history = model.fit(
    X_train_pad,
    y_train_enc,
    epochs=5,
    batch_size=128,
    validation_split=0.2
)

Epoch 1/5


I0000 00:00:1772005668.125568      67 cuda_dnn.cc:529] Loaded cuDNN version 91002


7274/7274 ━━━━━━━━━━━━━━━━━━━━ 77s 10ms/step - accuracy: 0.9434 - loss: 0.2174 - val_accuracy: 0.9502 - val_loss: 0.1494
Epoch 2/5
7274/7274 ━━━━━━━━━━━━━━━━━━━━ 74s 10ms/step - accuracy: 0.9511 - loss: 0.1469 - val_accuracy: 0.9521 - val_loss: 0.1396
Epoch 3/5
7274/7274 ━━━━━━━━━━━━━━━━━━━━ 74s 10ms/step - accuracy: 0.9536 - loss: 0.1345 - val_accuracy: 0.9541 - val_loss: 0.1309
Epoch 4/5
7274/7274 ━━━━━━━━━━━━━━━━━━━━ 74s 10ms/step - accuracy: 0.9558 - loss: 0.1255 - val_accuracy: 0.9547 - val_loss: 0.1301
Epoch 5/5
7274/7274 ━━━━━━━━━━━━━━━━━━━━ 74s 10ms/step - accuracy: 0.9578 - loss: 0.1188 - val_accuracy: 0.9549 - val_loss: 0.1260


In [13]:
loss, accuracy = model.evaluate(X_test_pad, y_test_enc)
print("LSTM Accuracy:", accuracy)

9092/9092 ━━━━━━━━━━━━━━━━━━━━ 37s 4ms/step - accuracy: 0.9551 - loss: 0.1251
LSTM Accuracy: 0.9550393223762512


In [14]:
import numpy as np

y_pred_probs = model.predict(X_test_pad)
y_pred = np.argmax(y_pred_probs, axis=1)

9092/9092 ━━━━━━━━━━━━━━━━━━━━ 23s 3ms/step


In [15]:
from sklearn.metrics import classification_report

print(classification_report(y_test_enc, y_pred, target_names=le.classes_))

              precision    recall  f1-score   support

    negative       0.64      0.62      0.63      6611
     neutral       0.39      0.29      0.34      9689
    positive       0.98      0.99      0.98    274643

    accuracy                           0.96    290943
   macro avg       0.67      0.63      0.65    290943
weighted avg       0.95      0.96      0.95    290943

